In [ ]:
# Colab GPU Smoke Test (single cell)
import os, sys, time, subprocess, textwrap

def sh(cmd):
    print(f"\n$ {cmd}")
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
        print(out.strip())
        return 0, out
    except subprocess.CalledProcessError as e:
        print(e.output.strip())
        return e.returncode, e.output

print("=== Python / Runtime ===")
print("python:", sys.executable)
print("version:", sys.version.split()[0])
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

print("\n=== GPU Presence (nvidia-smi) ===")
rc, _ = sh("nvidia-smi -L")
if rc != 0:
    print("\n❌ No GPU detected. In Colab: Runtime → Change runtime type → GPU")
else:
    sh("nvidia-smi")

print("\n=== CUDA Toolchain (optional) ===")
sh("nvcc --version")

print("\n=== PyTorch CUDA Check ===")
# Install torch if missing (Colab often has it, but this makes it robust)
try:
    import torch
except Exception:
    sh("pip -q install torch")
    import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

    # Warmup
    a = torch.randn((2048, 2048), device="cuda")
    b = torch.randn((2048, 2048), device="cuda")
    _ = a @ b
    torch.cuda.synchronize()

    # Timed matmul
    t0 = time.time()
    iters = 20
    for _ in range(iters):
        c = a @ b
    torch.cuda.synchronize()
    t1 = time.time()
    print(f"PyTorch GPU matmul: OK ({iters} iters)  total={t1-t0:.3f}s  per={(t1-t0)/iters:.4f}s")

    # Simple host->device and device->host transfer check
    x_cpu = torch.randn((64 * 1024 * 1024 // 4,), dtype=torch.float32)  # ~64MB
    t0 = time.time()
    x_gpu = x_cpu.to("cuda", non_blocking=False)
    torch.cuda.synchronize()
    t1 = time.time()
    y_cpu = x_gpu.to("cpu", non_blocking=False)
    torch.cuda.synchronize()
    t2 = time.time()
    print(f"H2D (~64MB): {(t1-t0):.3f}s, D2H: {(t2-t1):.3f}s")

else:
    print("\n❌ PyTorch does not see CUDA. Make sure Colab runtime is GPU and rerun the cell.")

print("\n=== TensorFlow GPU Check (optional) ===")
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    print("tensorflow:", tf.__version__)
    print("TF GPUs:", gpus)
    if gpus:
        with tf.device("/GPU:0"):
            A = tf.random.normal([2048, 2048])
            B = tf.random.normal([2048, 2048])
            _ = tf.matmul(A, B)
        print("TensorFlow GPU matmul: OK")
    else:
        print("TensorFlow sees no GPU (PyTorch CUDA above is the main check).")
except Exception as e:
    print("TensorFlow not installed or failed to import (fine):", repr(e))

print("\n✅ Done. If nvidia-smi lists a GPU and PyTorch CUDA matmul says OK, you're connected correctly.")

